# Customer Segmentation & Churn Pattern Analytics in European Banking
**Full analysis notebook** (Google Colab)  
Author: Suryapriya · Unified Mentor internship project

Run the cells from top to bottom.

## 1. Setup

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

plt.rcParams.update({"figure.figsize": (6, 4), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
BLUE, RED = "#2b6cb0", "#c53030"

## 2. Load the data
The cell tries GitHub first. If that fails (or the repo is private), it asks you to upload `European_Bank.csv`.

In [ ]:
URL = "https://raw.githubusercontent.com/SSuryapriya/european-bank-churn-analytics/main/European_Bank.csv"
try:
    df = pd.read_csv(URL)
    print("Loaded from GitHub")
except Exception:
    from google.colab import files
    up = files.upload()                      # choose European_Bank.csv
    df = pd.read_csv(list(up.keys())[0])
    print("Loaded from upload")
print(df.shape)
df.head()

## 3. Data validation

In [ ]:
print("Missing values :", df.isna().sum().sum())
print("Duplicate rows :", df.duplicated().sum())
print("CustomerId unique:", df.CustomerId.is_unique)
for c in ["HasCrCard", "IsActiveMember", "Exited"]:
    print(c, "->", sorted(df[c].unique()))
df.describe().T[["min", "max", "mean"]].round(1)

## 4. Cleaning and segment creation

In [ ]:
df = df.drop(columns=["Surname"])          # non-analytical field

df["Age_Group"] = pd.cut(df.Age, [0, 29, 45, 60, 200], labels=["<30", "30-45", "46-60", "60+"])
df["Tenure_Group"] = pd.cut(df.Tenure, [-1, 2, 6, 20], labels=["New (0-2)", "Mid (3-6)", "Long (7+)"])
df["Credit_Band"] = pd.cut(df.CreditScore, [0, 579, 669, 900], labels=["Low", "Medium", "High"])
median_bal = df.loc[df.Balance > 0, "Balance"].median()
df["Balance_Seg"] = pd.cut(df.Balance, [-1, 0, median_bal, 1e12], labels=["Zero", "Low", "High"])

HV_CUT = df.Balance.quantile(0.75)          # high-value = top 25% by balance
overall = df.Exited.mean() * 100
print(f"High-value cut-off: {HV_CUT:,.0f}")
df.head()

## 5. Overall churn and helper functions

In [ ]:
print(f"Customers: {len(df):,} | Churned: {df.Exited.sum():,} | Churn rate: {overall:.2f}%")

def churn_table(col, data=df):
    g = data.groupby(col, observed=True).Exited.agg(Customers="count", Churned="sum")
    g["Churn %"] = (g.Churned / g.Customers * 100).round(1)
    g["Share of all churn %"] = (g.Churned / g.Churned.sum() * 100).round(1)
    return g

def churn_bar(col, title):
    g = churn_table(col)
    colors = [RED if v == g["Churn %"].max() else BLUE for v in g["Churn %"]]
    ax = g["Churn %"].plot(kind="bar", color=colors, rot=0)
    ax.axhline(overall, ls="--", c="gray", lw=1)
    for i, v in enumerate(g["Churn %"]):
        ax.text(i, v + .8, f"{v:.1f}%", ha="center")
    ax.set_ylabel("Churn rate (%)"); ax.set_title(title)
    plt.tight_layout(); plt.show()
    return g

## 6. Geography

In [ ]:
g = churn_bar("Geography", "Churn rate by country")
g["Risk index"] = (g["Churn %"] / overall).round(2)   # Geographic Risk Index
g

## 7. Age, and geography x age

In [ ]:
churn_bar("Age_Group", "Churn rate by age group")

heat = df.pivot_table(index="Geography", columns="Age_Group", values="Exited",
                      aggfunc="mean", observed=True) * 100
fig, ax = plt.subplots(figsize=(6, 3.5))
im = ax.imshow(heat.values, cmap="Reds")
ax.set_xticks(range(heat.shape[1]), heat.columns); ax.set_yticks(range(heat.shape[0]), heat.index)
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        ax.text(j, i, f"{heat.values[i, j]:.0f}%", ha="center", va="center")
ax.set_title("Churn % by country and age group"); fig.colorbar(im); plt.tight_layout(); plt.show()

## 8. Gender

In [ ]:
churn_bar("Gender", "Churn rate by gender")
(df.pivot_table(index="Gender", columns="Geography", values="Exited") * 100).round(1)

## 9. Products and engagement

In [ ]:
churn_bar("NumOfProducts", "Churn rate by number of products")
act = churn_bar("IsActiveMember", "Churn rate by activity (0 = inactive, 1 = active)")
print("Engagement Drop Indicator (inactive / active):", round(act.loc[0, "Churn %"] / act.loc[1, "Churn %"], 2))
(df.pivot_table(index="IsActiveMember", columns="NumOfProducts", values="Exited") * 100).round(1)

## 10. Tenure, credit score and balance

In [ ]:
churn_bar("Tenure_Group", "Churn rate by tenure")
churn_bar("Credit_Band", "Churn rate by credit score band")
churn_bar("Balance_Seg", "Churn rate by balance segment")

## 11. Churned vs retained profile

In [ ]:
cols = ["CreditScore", "Age", "Tenure", "Balance", "NumOfProducts", "EstimatedSalary", "IsActiveMember"]
prof = df.groupby("Exited")[cols].mean().round(1).T
prof.columns = ["Retained", "Churned"]
prof

## 12. High-value customers

In [ ]:
hv = df[df.Balance >= HV_CUT]
rest = df[df.Balance < HV_CUT]
total_lost = df.loc[df.Exited == 1, "Balance"].sum()
hv_lost = hv.loc[hv.Exited == 1, "Balance"].sum()

print(f"High-value customers : {len(hv):,}")
print(f"High-Value Churn Ratio: {hv.Exited.mean()*100:.1f}%  (rest: {rest.Exited.mean()*100:.1f}%)")
print(f"Balance lost (all churners)  : {total_lost/1e6:.1f}M")
print(f"Balance lost (HV churners)   : {hv_lost/1e6:.1f}M  ({hv_lost/total_lost*100:.1f}% of total)")
print("\nHV churn by country (%):\n", (hv.groupby("Geography").Exited.mean() * 100).round(1))
print("\nHV churn by activity (%):\n", (hv.groupby("IsActiveMember").Exited.mean() * 100).round(1))
print("\nCorrelation with churn -> salary:", round(df.EstimatedSalary.corr(df.Exited), 3),
      "| balance:", round(df.Balance.corr(df.Exited), 3))

for e, c, l in [(0, BLUE, "Retained"), (1, RED, "Churned")]:
    s = hv[hv.Exited == e]; plt.scatter(s.EstimatedSalary, s.Balance, s=6, alpha=.45, c=c, label=l)
plt.xlabel("Estimated salary"); plt.ylabel("Balance"); plt.legend(frameon=False, markerscale=3)
plt.title("Salary vs balance (high-value customers)"); plt.tight_layout(); plt.show()

## 13. KPI summary

In [ ]:
kpi = pd.DataFrame({
    "KPI": ["Overall Churn Rate", "High-Value Churn Ratio", "Geographic Risk Index (Germany)",
            "Engagement Drop Indicator"],
    "Value": [f"{overall:.1f}%", f"{hv.Exited.mean()*100:.1f}%",
              round(churn_table("Geography").loc["Germany", "Churn %"] / overall, 2),
              round(act.loc[0, "Churn %"] / act.loc[1, "Churn %"], 2)]})
kpi

## 14. Statistical tests (chi-square)
A small p-value (< 0.05) means the segment is significantly related to churn.

In [ ]:
rows = []
for c in ["Geography", "Gender", "Age_Group", "Tenure_Group", "Credit_Band", "Balance_Seg",
          "NumOfProducts", "IsActiveMember", "HasCrCard"]:
    p = chi2_contingency(pd.crosstab(df[c], df.Exited))[1]
    rows.append((c, p, "Significant" if p < 0.05 else "Not significant"))
pd.DataFrame(rows, columns=["Variable", "p-value", "Result"]).round(4)

## 15. What drives churn? (random forest)

In [ ]:
X = pd.get_dummies(df[["CreditScore", "Geography", "Gender", "Age", "Tenure", "Balance",
                       "NumOfProducts", "HasCrCard", "IsActiveMember", "EstimatedSalary"]],
                   columns=["Geography", "Gender"], drop_first=True)
y = df.Exited
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.25, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced",
                            random_state=42).fit(Xtr, ytr)
print("Accuracy:", round(accuracy_score(yte, rf.predict(Xte)), 3),
      "| AUC:", round(roc_auc_score(yte, rf.predict_proba(Xte)[:, 1]), 3))

imp = pd.Series(rf.feature_importances_ * 100, X.columns).sort_values()
imp.plot(kind="barh", color=BLUE); plt.xlabel("Importance (%)")
plt.title("What drives churn?"); plt.tight_layout(); plt.show()

## 16. Export (optional)
Saves the cleaned, segmented data so you can download it.

In [ ]:
df.to_csv("churn_segmented.csv", index=False)
from google.colab import files
files.download("churn_segmented.csv")